In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
df=pd.read_csv('/content/spam.csv')
df.sample(2)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
5534,ham,Ok which your another number,NaN,NaN,NaN
5207,ham,"Babe, I'm answering you, can't you see me ? Ma...",NaN,NaN,NaN


In [ ]:
df=df.drop(['Unnamed: 2','Unnamed: 3','Unnamed: 4'],axis=1)
df=df.rename(columns={'v1':'target','v2':'text'})
df['target_enc']=df['target'].map({'ham':0,'spam':1})

In [ ]:
df

,target,text,target_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will Ã¼ b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


use this only when needed

In [ ]:
# import spacy
# nlp=spacy.blank('en')
# df['tokens'] = df['text'].apply(nlp)

In [ ]:
import string

In [ ]:
def tokenize(text):
  text=text.lower()
  text=text.replace("\n"," ").replace("\t"," ")
  translator=text.maketrans(","," ",string.punctuation)
  text=text.translate(translator)
  tokens=text.split()
  return tokens

In [ ]:
df['tokens']=df['text'].apply(tokenize)

In [ ]:
df

,target,text,target_enc,tokens
0,ham,"Go until jurong point, crazy.. Available only ...",0,"[go, until, jurong, point, crazy, available, o..."
1,ham,Ok lar... Joking wif u oni...,0,"[ok, lar, joking, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,ham,U dun say so early hor... U c already then say...,0,"[u, dun, say, so, early, hor, u, c, already, t..."
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,"[nah, i, dont, think, he, goes, to, usf, he, l..."
...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1,"[this, is, the, 2nd, time, we, have, tried, 2,..."
5568,ham,Will Ã¼ b going to esplanade fr home?,0,"[will, ã¼, b, going, to, esplanade, fr, home]"
5569,ham,"Pity, * was in mood for that. So...any other s...",0,"[pity, was, in, mood, for, that, soany, other,..."
5570,ham,The guy did some bitching but I acted like i'd...,0,"[the, guy, did, some, bitching, but, i, acted,..."


In [ ]:
df['target'].value_counts()

,count
target,
ham,4825
spam,747


In [ ]:
from sklearn.model_selection import train_test_split
X_temp,X_test,y_temp,y_test=train_test_split(df['tokens'],df['target_enc'],test_size=0.15,stratify=df['target_enc'],random_state=42)

In [ ]:
X_train,X_val,y_train,y_val=train_test_split(X_temp,y_temp,test_size=0.176,stratify=y_temp,random_state=42)

In [ ]:
X_train.shape,X_val.shape,X_test.shape,y_train.shape,y_val.shape,y_test.shape

((3902,), (834,), (836,), (3902,), (834,), (836,))

In [ ]:
print(y_train.value_counts())
print(y_val.value_counts())
print(y_test.value_counts())

target_enc
0    3379
1     523
Name: count, dtype: int64
target_enc
0    722
1    112
Name: count, dtype: int64
target_enc
0    724
1    112
Name: count, dtype: int64


In [ ]:
from collections import Counter

In [ ]:
def vocab_count(token_list,max_count=5000):
  counter=Counter()
  for token in token_list:
    counter.update(token)
  most_common=counter.most_common(max_count)
  vocab={'<UNK>':1,'<PAD>':0}
  for word,_ in most_common:
    if word not in vocab:
      vocab[word]=len(vocab)
  return vocab

In [ ]:
vocab=vocab_count(X_train,max_count=5000)
print(len(vocab))

5002


In [ ]:
print(list(vocab.items())[:10])


[('<UNK>', 1), ('<PAD>', 0), ('i', 2), ('to', 3), ('you', 4), ('a', 5), ('the', 6), ('u', 7), ('and', 8), ('is', 9)]


In [ ]:
print(X_train.iloc[2])

['hope', 'youâ\x92re', 'not', 'having', 'too', 'much', 'fun', 'without', 'me', 'see', 'u', 'tomorrow', 'love', 'jess', 'x']


In [ ]:
def text_to_indices(tokens_list,vocab):
  all_indices=[]
  for token in tokens_list:
    if token in vocab:
      all_indices.append(vocab[token])
    else:
      all_indices.append(vocab['<UNK>'])
  return all_indices

In [ ]:
X_train_indices = X_train.apply(lambda tokens: text_to_indices(tokens, vocab))
X_val_indices = X_val.apply(lambda tokens: text_to_indices(tokens, vocab))
X_test_indices = X_test.apply(lambda tokens: text_to_indices(tokens, vocab))

In [ ]:
X_train.iloc[1]

['then', 'i', 'buy']

In [ ]:
X_train_indices.iloc[1]

[58, 2, 185]

In [ ]:
lengths = df['text'].apply(lambda x: len(x.split()))
# max_len=lengths.max()
max_length = int(np.percentile(lengths, 99))

In [ ]:
max_length

55

In [ ]:
def pad_seq(seq,max_len):
  padded_seq=[]
  for s in seq:
    if len(s)<max_len:
      s=s+[0]*(max_len-len(s))
    else:
      s=s[:max_len]

    padded_seq.append(s)
  return padded_seq

In [ ]:
MAX_LEN=max_length
train_padded=pad_seq(X_train_indices,max_len=MAX_LEN)
val_padded=pad_seq(X_val_indices,max_len=MAX_LEN)
test_padded=pad_seq(X_test_indices,max_len=MAX_LEN)

In [ ]:
print(train_padded[3200])
print(test_padded[835])
print(val_padded[833])

[155, 4, 61, 3, 85, 12, 321, 504, 95, 22, 26, 633, 95, 176, 2286, 1, 10, 6, 727, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[319, 54, 1105, 4, 51, 1, 3410, 4, 356, 186, 55, 119, 16, 9, 150, 305, 5, 17, 19, 1, 45, 307, 1, 3072, 279, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[594, 14, 865, 362, 627, 13, 325, 863, 685, 1063, 561, 17, 1, 686, 1, 628, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
x=[1,2,3,4,5,5,66,677775,2]
len(x)

9

In [ ]:
from torch.utils.data import Dataset,DataLoader

In [ ]:
class SMSDataset(Dataset):
  def __init__(self,X,y):
    self.X=X
    self.y=y

  def __len__(self):
    return len(self.y)

  def __getitem__(self,idx):
    x_tensor=torch.tensor(self.X[idx],dtype=torch.long)
    y_tensor=torch.tensor(self.y[idx],dtype=torch.float) # Changed dtype to float
    return x_tensor,y_tensor

In [ ]:
y_train=y_train.tolist()
y_val=y_val.tolist()
y_test=y_test.tolist()

In [ ]:
train_dataset=SMSDataset(train_padded,y_train)
val_dataset=SMSDataset(val_padded,y_val)
test_dataset=SMSDataset(test_padded,y_test)

In [ ]:
x_sample,y_sample=train_dataset[0]
print(x_sample)
print(y_sample)

tensor([1218,   64,  210, 1077,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0])
tensor(0.)


In [ ]:
class DenseTextModel(nn.Module):
  def __init__(self,vocab_size,embed_dim,hidden_dim,padding_idx=0):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=embed_dim,padding_idx=padding_idx)
    self.fc1=nn.Linear(embed_dim,hidden_dim)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(hidden_dim,1)
    # Removed the sigmoid here; model will output logits
  def forward(self,x):
    embed=self.embedding(x)
    pooled=embed.mean(dim=1)#global average pooling
    layer1=self.fc1(pooled)
    relu=self.relu(layer1)
    layer2=self.fc2(relu)
    # Return raw logits
    return layer2

In [ ]:
x=train_dataset[0]
embed=nn.Embedding(len(x[0]),128,0)
print(embed.weight.shape)

torch.Size([55, 128])


In [ ]:
layer1=nn.Linear(128,64)
layer1.weight.shape

torch.Size([64, 128])

In [ ]:
layer2=nn.Linear(64,1)
layer2.weight.shape

torch.Size([1, 64])

In [ ]:
neg_count = y_train.count(0)
pos_count = y_train.count(1)
pos_count

523

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model1=DenseTextModel(vocab_size=len(vocab), embed_dim=128, hidden_dim=64, padding_idx=0).to(device)

# Calculate pos_weight for imbalanced dataset
neg_count = y_train.count(0)
pos_count = y_train.count(1)
pos_weight_tensor = torch.tensor([neg_count / pos_count], device=device, dtype=torch.float)
loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

optimizer=optim.Adam(model1.parameters(),lr=1e-3)

In [ ]:
#training loop
def train_one_epoch(model,loss_fn,optimizer,dataloader,device):
  model.train()
  total_loss=0
  for inputs,labels in dataloader:
    inputs=inputs.to(device)
    labels=labels.to(device).unsqueeze(1) # Ensure labels are [batch_size, 1]
    optimizer.zero_grad()
    outputs=model(inputs) # Model now outputs logits
    loss=loss_fn(outputs,labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  return total_loss/len(dataloader)

In [ ]:

def val_one_epoch(model,loss_fn,dataloader,device):
  model.eval()
  total_loss=0
  correct=0
  total=0
  with torch.inference_mode():
    for inputs,labels in dataloader:
      inputs=inputs.to(device)
      labels=labels.to(device).unsqueeze(1) # Ensure labels are [batch_size, 1]
      outputs=model(inputs) # Model outputs logits
      loss=loss_fn(outputs,labels)

      # Apply sigmoid to logits to get probabilities for prediction thresholding
      probs = torch.sigmoid(outputs)
      preds=(probs >=0.5).float()

      correct +=(preds ==labels).sum().item()
      total +=labels.size(0)
      total_loss+=loss.item()
    return total_loss/len(dataloader),correct/total

In [ ]:
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
sample_inputs, sample_labels = next(iter(train_loader))

# Move inputs to the device the model is on
sample_inputs = sample_inputs.to(device)

# Pass the sample inputs through the model
model_output = model1(sample_inputs)

print(f"Shape of the model's output (after sigmoid): {model_output.shape}")

Shape of the model's output (after sigmoid): torch.Size([32, 1])


In [ ]:
epochs=10
for epoch in range(epochs):
  train_loss=train_one_epoch(model1,loss_fn,optimizer,train_loader,device)
  val_loss,val_acc=val_one_epoch(model1,loss_fn,val_loader,device)


  print(f"""Epoch [{epoch+1}/{epochs}] Train Loss: {train_loss:.4f}
Val Loss:   {val_loss:.4f} Val Acc:    {(val_acc*100):.4f}%""") # Display accuracy as percentage

Epoch [1/10] Train Loss: 1.0122
Val Loss:   0.6599 Val Acc:    83.9329%
Epoch [2/10] Train Loss: 0.4433
Val Loss:   0.3145 Val Acc:    94.7242%
Epoch [3/10] Train Loss: 0.2463
Val Loss:   0.2447 Val Acc:    97.4820%
Epoch [4/10] Train Loss: 0.1676
Val Loss:   0.2193 Val Acc:    97.2422%
Epoch [5/10] Train Loss: 0.1163
Val Loss:   0.2326 Val Acc:    94.9640%
Epoch [6/10] Train Loss: 0.0872
Val Loss:   0.2126 Val Acc:    97.4820%
Epoch [7/10] Train Loss: 0.0674
Val Loss:   0.2145 Val Acc:    97.7218%
Epoch [8/10] Train Loss: 0.0533
Val Loss:   0.2195 Val Acc:    97.4820%
Epoch [9/10] Train Loss: 0.0433
Val Loss:   0.2274 Val Acc:    97.8417%
Epoch [10/10] Train Loss: 0.0345
Val Loss:   0.2508 Val Acc:    97.8417%


In [ ]:
  from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [ ]:
def model_metrics(model,dataloader,device):
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for inputs, labels in dataloader:
        inputs = inputs.to(device)

        outputs = model(inputs)
        preds = (outputs >= 0.5).int().cpu().numpy()

        all_preds.extend(preds.flatten())
        all_labels.extend(labels.cpu().numpy().flatten())
  return all_labels,all_preds

In [ ]:
all_labels,all_preds=model_metrics(model1,val_loader,device)

In [ ]:

precision = precision_score(all_labels, all_preds)
recall    = recall_score(all_labels, all_preds)
f1        = f1_score(all_labels, all_preds)
accuracy  = accuracy_score(all_labels, all_preds)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Accuracy:", accuracy)


Precision: 0.9298245614035088
Recall: 0.9464285714285714
F1: 0.9380530973451328
Accuracy: 0.9832134292565947


In [ ]:

cm = confusion_matrix(all_labels, all_preds)
print(cm)


[[714   8]
 [  6 106]]


In [ ]:
def test_one_epoch(model,loss_fn,dataloader,device):
  model.eval()
  total_loss=0
  correct=0
  total=0
  with torch.inference_mode():
    for inputs,labels in dataloader:
      inputs=inputs.to(device)
      labels=labels.to(device).unsqueeze(1) # Ensure labels are [batch_size, 1]
      outputs=model(inputs) # Model outputs logits
      loss=loss_fn(outputs,labels)

      # Apply sigmoid to logits to get probabilities for prediction thresholding
      probs = torch.sigmoid(outputs)
      preds=(probs >=0.5).float()

      correct +=(preds ==labels).sum().item()
      total +=labels.size(0)
      total_loss+=loss.item()
    return total_loss/len(dataloader),correct/total

In [ ]:
test_loss,test_acc=test_one_epoch(model1,loss_fn,test_loader,device)
print(f"""Test Loss:   {test_loss:.4f} Test Acc:    {(test_acc*100):.4f}%""")

Test Loss:   0.3112 Test Acc:    97.6077%


In [ ]:

all_labels,all_preds=model_metrics(model1,test_loader,device)

In [ ]:

precision = precision_score(all_labels, all_preds)
recall    = recall_score(all_labels, all_preds)
f1        = f1_score(all_labels, all_preds)
accuracy  = accuracy_score(all_labels, all_preds)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Accuracy:", accuracy)

Precision: 0.9345794392523364
Recall: 0.8928571428571429
F1: 0.91324200913242
Accuracy: 0.9772727272727273


In [ ]:

cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[717   7]
 [ 12 100]]


In [ ]:
class bilstmmodel(nn.Module):
  def __init__(self,vocab_size,embed_dim,hidden_dim,padding_idx=0):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=embed_dim,padding_idx=padding_idx)
    self.lstm1=nn.LSTM(embed_dim,hidden_dim,batch_first=True,bidirectional=True,dropout=0.0,num_layers=1)
    self.lstm2=nn.LSTM(hidden_dim*2,hidden_dim,batch_first=True,bidirectional=True,num_layers=1)
    self.dropout=nn.Dropout(0.1)
    self.fc1=nn.Linear(hidden_dim*2,32)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(32,1)
  def forward(self,x):
    x=self.embedding(x)
    x,_=self.lstm1(x)
    x,_=self.lstm2(x)
    x=self.dropout(x)
    x = x.mean(dim=1) # Global average pooling to aggregate sequence dimension
    x=self.relu(self.fc1(x))
    return self.fc2(x)

In [ ]:
x=train_dataset[0]
embed=nn.Embedding(len(x[0]),128,0)
print(embed.weight.shape)
lstm1=nn.LSTM(128,64,batch_first=True,bidirectional=True,dropout=0.0,num_layers=1)
print(lstm1.weight_ih_l0.shape)
lstm2=nn.LSTM(64*2,64,batch_first=True,bidirectional=True,num_layers=1)
print(lstm2.weight_ih_l0.shape)
fc1=nn.Linear(64*2,32)
print(layer1.weight.shape)
layer2=nn.Linear(32,1)
print(layer2.weight.shape)

torch.Size([55, 128])
torch.Size([256, 128])
torch.Size([256, 128])
torch.Size([64, 128])
torch.Size([1, 32])


In [ ]:
model2=bilstmmodel(vocab_size=len(vocab),embed_dim=128,hidden_dim=64,padding_idx=0)
# Calculate pos_weight for imbalanced dataset
neg_count = y_train.count(0)
pos_count = y_train.count(1)
pos_weight_tensor = torch.tensor([neg_count / pos_count], device=device, dtype=torch.float)
loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer=optim.Adam(model2.parameters(),lr=1e-3)

In [ ]:
#training loop
def train_one_epoch(model,loss_fn,optimizer,dataloader,device):
  model.train()
  total_loss=0
  for inputs,labels in dataloader:
    inputs=inputs.to(device)
    labels=labels.to(device).unsqueeze(1) # Ensure labels are [batch_size, 1]
    optimizer.zero_grad()
    outputs=model(inputs) # Model now outputs logits
    loss=loss_fn(outputs,labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  return total_loss/len(dataloader)

In [ ]:
def val_one_epoch(model,loss_fn,dataloader,device):
  model.eval()
  total_loss=0
  correct=0
  total=0
  with torch.inference_mode():
    for inputs,labels in dataloader:
      inputs=inputs.to(device)
      labels=labels.to(device).unsqueeze(1) # Ensure labels are [batch_size, 1]
      outputs=model(inputs) # Model outputs logits
      loss=loss_fn(outputs,labels)

      # Apply sigmoid to logits to get probabilities for prediction thresholding
      probs = torch.sigmoid(outputs)
      preds=(probs >=0.5).float()

      correct +=(preds ==labels).sum().item()
      total +=labels.size(0)
      total_loss+=loss.item()
    return total_loss/len(dataloader),correct/total

In [ ]:
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
sample_inputs, sample_labels = next(iter(train_loader))

# Move inputs to the device the model is on
sample_inputs = sample_inputs.to(device)

# Pass the sample inputs through the model
model_output = model2(sample_inputs)

print(f"Shape of the model's output (after sigmoid): {model_output.shape}")

Shape of the model's output (after sigmoid): torch.Size([32, 1])


In [ ]:
epochs=10
for epoch in range(epochs):
  train_loss=train_one_epoch(model2,loss_fn,optimizer,train_loader,device)
  val_loss,val_acc=val_one_epoch(model2,loss_fn,val_loader,device)


  print(f"""Epoch [{epoch+1}/{epochs}] Train Loss: {train_loss:.4f}
Val Loss:   {val_loss:.4f} Val Acc:    {(val_acc*100):.4f}%""") # Display accuracy as percentage

Epoch [1/10] Train Loss: 0.0628
Val Loss:   0.1982 Val Acc:    95.9233%
Epoch [2/10] Train Loss: 0.0399
Val Loss:   0.1905 Val Acc:    98.0815%
Epoch [3/10] Train Loss: 0.0221
Val Loss:   0.1951 Val Acc:    97.9616%
Epoch [4/10] Train Loss: 0.0170
Val Loss:   0.2425 Val Acc:    98.5612%
Epoch [5/10] Train Loss: 0.0162
Val Loss:   0.2634 Val Acc:    98.4412%
Epoch [6/10] Train Loss: 0.0155
Val Loss:   0.2489 Val Acc:    98.2014%
Epoch [7/10] Train Loss: 0.0137
Val Loss:   0.3121 Val Acc:    98.2014%
Epoch [8/10] Train Loss: 0.0136
Val Loss:   0.3063 Val Acc:    98.0815%
Epoch [9/10] Train Loss: 0.0160
Val Loss:   0.3007 Val Acc:    94.7242%
Epoch [10/10] Train Loss: 0.0381
Val Loss:   0.2113 Val Acc:    98.2014%


In [ ]:
  from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [ ]:
def model_metrics(model,dataloader,device):
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for inputs, labels in dataloader:
        inputs = inputs.to(device)

        outputs = model(inputs)
        preds = (outputs >= 0.5).int().cpu().numpy()

        all_preds.extend(preds.flatten())
        all_labels.extend(labels.cpu().numpy().flatten())
  return all_labels,all_preds

In [ ]:
all_labels,all_preds=model_metrics(model2,val_loader,device)
precision = precision_score(all_labels, all_preds)
recall    = recall_score(all_labels, all_preds)
f1        = f1_score(all_labels, all_preds)
accuracy  = accuracy_score(all_labels, all_preds)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Accuracy:", accuracy)

cm = confusion_matrix(all_labels, all_preds)
print(cm)


Precision: 0.9298245614035088
Recall: 0.9464285714285714
F1: 0.9380530973451328
Accuracy: 0.9832134292565947
[[714   8]
 [  6 106]]


In [ ]:
def test_one_epoch(model,loss_fn,dataloader,device):
  model.eval()
  total_loss=0
  correct=0
  total=0
  with torch.inference_mode():
    for inputs,labels in dataloader:
      inputs=inputs.to(device)
      labels=labels.to(device).unsqueeze(1) # Ensure labels are [batch_size, 1]
      outputs=model(inputs) # Model outputs logits
      loss=loss_fn(outputs,labels)

      # Apply sigmoid to logits to get probabilities for prediction thresholding
      probs = torch.sigmoid(outputs)
      preds=(probs >=0.5).float()

      correct +=(preds ==labels).sum().item()
      total +=labels.size(0)
      total_loss+=loss.item()
    return total_loss/len(dataloader),correct/total

In [ ]:
test_loss,test_acc=test_one_epoch(model2,loss_fn,test_loader,device)
print(f"""Test Loss:   {test_loss:.4f} Test Acc:    {(test_acc*100):.4f}%""")
all_labels,all_preds=model_metrics(model2,test_loader,device)
precision = precision_score(all_labels, all_preds)
recall    = recall_score(all_labels, all_preds)
f1        = f1_score(all_labels, all_preds)
accuracy  = accuracy_score(all_labels, all_preds)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Accuracy:", accuracy)
cm = confusion_matrix(all_labels, all_preds)
print(cm)

Test Loss:   0.4277 Test Acc:    97.9665%
Precision: 0.9439252336448598
Recall: 0.9017857142857143
F1: 0.9223744292237442
Accuracy: 0.9796650717703349
[[718   6]
 [ 11 101]]
